****DICOM > NIfTI****

In [ ]:
# Install Apptainer
sudo add-apt-repository -y ppa:apptainer/ppa
sudo apt update
sudo apt install -y apptainer

# Check if it worked
apptainer --version

In [ ]:
# Download Heudiconv
singularity pull heudiconv.sif docker://nipy/heudiconv:latest

In [ ]:
# Get your DICOM data
# I used: https://github.com/datalad/example-dicom-functional/tree/master/dicoms
git clone https://github.com/datalad/example-dicom-functional.git
cd example-dicom-functional

In [ ]:
# Run heudiconv
# Before running the actual conversion, we set up the environment
singularity run --cleanenv -B $(pwd):/base heudiconv.sif -s 02 -f convertall -c none -o /base/Nifti --files /base/dicoms

# Check what it finds
find . -iname "*.tsv"
cat Nifti/.heudiconv/02/info/dicominfo.tsv

In [ ]:
# Write the heuristic
mkdir -p code
cat > code/heuristic.py << 'EOF'
def create_key(template):
    return template, ('nii.gz',), None

def infotodict(seqinfo):
    func = create_key('sub-{subject}/func/sub-{subject}_task-oneback_run-{item:02d}_bold')
    info = {func: []}

    for s in seqinfo:
        if 'func_task-oneback' in s.protocol_name:
            info[func].append(s.series_id)

    return info
EOF

In [ ]:
# Run heudiconv
rm -rf Nifti
singularity run --cleanenv -B $(pwd):/base heudiconv.sif -s 02 -f /base/code/heuristic.py -c dcm2niix -b -o /base/Nifti --files /base/dicoms
# Check the results
find Nifti -iname "*.nii.gz"

###### Conversion Draft Script - Mina R.Jam ######